# Customer Churn Analysis & Model Experimentation

This notebook covers the Exploratory Data Analysis (EDA) and Model Experimentation phases of the **Customer Churn Prediction System**.

## Objectives
1. Load and understand the synthetic customer dataset.
2. Perform Exploratory Data Analysis to identify key churn drivers.
3. Visualize relationships between user behavior, satisfaction, and churn.
4. Train and compare multiple classification models (Logistic Regression, Decision Tree, Random Forest).
5. Select the best performing model based on Accuracy and F1-score on the churned class.

In [ ]:
# Set up paths to import local modules
import os
import sys
sys.path.append(os.path.abspath('../src'))

# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import Image, display

# Import local pipeline scripts
from data_generation import generate_synthetic_data
from preprocessing import ChurnPreprocessor
from eda_visualization import run_eda
from model_training import train_and_evaluate

## 1. Load Dataset

We load the synthetic customer dataset generated at `data/customer_churn_data.csv`. If it doesn't exist, we generate it on the fly.

In [ ]:
csv_path = "../data/customer_churn_data.csv"
if not os.path.exists(csv_path):
    print("Dataset CSV not found. Generating new dataset...")
    df = generate_synthetic_data()
    os.makedirs("../data", exist_ok=True)
    df.to_csv(csv_path, index=False)
else:
    df = pd.read_csv(csv_path)

print(f"Loaded dataset with shape: {df.shape}")
df.head()

## 2. Exploratory Data Analysis & Visualizations

We run the EDA visualization module which generates our required charts and saves them to `../plots/`.

In [ ]:
# Run EDA and save plots
run_eda(df, plots_dir="../plots")

### Display Saved Visualizations

Let's load and inspect the generated charts inline.

In [ ]:
print("### Churn Distribution (Pie Chart)")
display(Image(filename='../plots/churn_distribution_pie.png'))

print("\n### Churn Rate by Subscription Type")
display(Image(filename='../plots/churn_rate_by_subscription.png'))

print("\n### Feature Correlation Matrix")
display(Image(filename='../plots/correlation_matrix_heatmap.png'))

print("\n### Monthly Charges Distribution")
display(Image(filename='../plots/monthly_charges_histogram.png'))

print("\n### Tenure and Satisfaction Boxplots")
display(Image(filename='../plots/tenure_satisfaction_boxplots.png'))

## 3. Model Training & Evaluation

We train, compare, and serialize our classification models using the training pipeline.

In [ ]:
# Execute training pipeline
comparison_df = train_and_evaluate(csv_path=csv_path, models_dir="../models")

## 4. Feature Importance Inspection

Let's inspect the coefficients of our best performing model (Logistic Regression) to understand what factors impact customer churn the most.

In [ ]:
import joblib

# Load best model and preprocessor
best_model = joblib.load("../models/churn_model.pkl")
preprocessor = joblib.load("../models/preprocessor.pkl")

if hasattr(best_model, "coef_"):
    # Get feature names
    cat_encoded_cols = preprocessor.encoder.get_feature_names_out(preprocessor.cat_cols)
    feature_names = preprocessor.num_cols + list(cat_encoded_cols)
    
    # Match coefficients to feature names
    coefs = best_model.coef_[0]
    importance_df = pd.DataFrame({
        'Feature': feature_names,
        'Coefficient': coefs,
        'Absolute Importance': np.abs(coefs)
    }).sort_values(by='Absolute Importance', ascending=False)
    
    # Plot coefficients
    plt.figure(figsize=(10, 6))
    sns.barplot(x='Coefficient', y='Feature', data=importance_df, palette='viridis')
    plt.title('Feature Impact on Churn (Logistic Regression Coefficients)', fontsize=14, weight='bold')
    plt.xlabel('Coefficient Value (Positive = Drives Churn, Negative = Prevents Churn)')
    plt.ylabel('Feature')
    plt.tight_layout()
    plt.savefig('../plots/feature_coefficients.png', dpi=150)
    plt.show()
    
    print("### Feature Importance Table:")
    display(importance_df)
else:
    print("Best model does not support coef_ attribute.")